# Step 12 — assign population and select a BEAM subdivision

**# of cells in notebook:** 2

**Purpose:** Assign population to the alternative BEAM partitions created in Step 11 and select one BEAM subdivision for each source block. The selected BEAM solution replaces the standard k-based subdivision selected in Step 9 for blocks that were manually routed through the BEAM workflow.

**Input:**

- `graph_boundary_penalty_partitions_beam.gpkg` from Step 11, containing available:
  - `selected_dissolved_n2`
  - `selected_dissolved_n3`
  - `selected_dissolved_n4`
- each BEAM candidate layer includes summed `cell_area_m2`, representing the total tessellation-cell area of that candidate new block
- a population geodatabase containing:
  - `pop_grid`
  - `buildings_inside`
- `heterogeneous_largePop_blocks` from Step 1
- the manually specified set of blocks being processed through BEAM

**Output:**

Within each BEAM-processing block folder:

- `new_blocks_populated_beam.gpkg`
- populated versions of the available `selected_dissolved_n2`, `n3`, and `n4` layers
- per-layer population-assignment and grid-apportionment CSVs

At the base block directory:

- `new_blocks_population_assignment_summary_beam_no_arcpy.csv`

Within `heterogeneous_largePop_selection`:

- `new_blocks_populated.gpkg` — the canonical selected subdivision; replaced by the chosen BEAM solution for BEAM-processed blocks
- `new_blocks_populated_pre_beam.gpkg` — preserved Step 9 selection, when present

At the selection-directory level:

- `beam_selection_log.txt`
- `beam_selection_summary.csv`

**Main logic:**

**Cell 1 — Assign population to BEAM candidate partitions**

1. Processes the manually selected BEAM source blocks from Step 11.
2. Reads the available n = 2, 3, and 4 dissolved BEAM candidate layers.
3. Selects relevant population-grid cells and buildings.
4. Intersects buildings with grid cells and candidate block features.
5. Apportions grid-cell population among candidate blocks according to their shares of building footprint area.
6. Adds `population` to each BEAM candidate feature.
7. Writes populated candidate layers and detailed QA outputs.

**Cell 2 — Select the preferred BEAM solution**

1. Reads the source block's LargePop and heterogeneity flags.
2. Evaluates the available populated n = 2–4 BEAM candidates.
3. For `LargePop = 1`, selects the first n where every new block has population below 1,000.
4. If none satisfy the LargePop population criterion, evaluates n = 4 against the 100,000 m² threshold using summed `cell_area_m2`; otherwise uses n = 4 with a warning.
5. For non-LargePop heterogeneous blocks, selects the first n where every feature has `cell_area_m2` below 100,000 m²; if none pass, uses n = 4 and records a warning.
6. Preserves the Step 9 selection as `new_blocks_populated_pre_beam.gpkg`, writes the chosen BEAM layer to the canonical `new_blocks_populated.gpkg`, and records the decision in `beam_selection_summary.csv`.

`cell_area_m2`, rather than building footprint area, is the area measure used for the 100,000 m² candidate-block threshold.


In [ ]:
# -*- coding: utf-8 -*-
"""
Assign population to BEAM partition block layers WITHOUT ArcPy.

This version processes selected block folders only and reads layers from:

    graph_boundary_penalty_partitions_beam.gpkg

Expected input layers, as seen by ArcGIS:
    main.selected_dissolved_n2
    main.selected_dissolved_n3
    main.selected_dissolved_n4

Expected input layers, as seen by pyogrio/GDAL:
    selected_dissolved_n2
    selected_dissolved_n3
    selected_dissolved_n4

Outputs:
    <block_folder>\\new_blocks_populated_beam.gpkg

No ArcPy required.
"""

import re
import csv
import traceback
from pathlib import Path
from collections import defaultdict

import pandas as pd
import geopandas as gpd
import pyogrio


# ============================================================
# USER SETTINGS
# ============================================================

large_pop_blocks_folder = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"
)

population_gdb = Path(
    r"E:\_johannesburg\_analysis\population_wp2\population_wp2.gdb"
)

# Only process these folders
block_filter = [
    "_52730",
    "_56219",
    "_62733",
    "_71608",
    "_72054",
    "_76290",
    "_83562",
    "_84830",
    "_89704"
]

# Input/output GeoPackages
input_blocks_gpkg_name = "graph_boundary_penalty_partitions_beam.gpkg"
populated_blocks_gpkg_name = "new_blocks_populated_beam.gpkg"

# Layers to process from graph_boundary_penalty_partitions_beam.gpkg
# Use the non-main names here; the helper below will also tolerate main.<name>.
beam_layer_names = [
    "selected_dissolved_n2",
    "selected_dissolved_n3",
    "selected_dissolved_n4",
]

pop_grid_layer = "pop_grid"
buildings_layer = "buildings_inside"

population_field = "population"
area_field = "area_m_utm_new"
pop_field = "grid_code"

overwrite_outputs = True
fix_invalid_geometries = False
area_tolerance = 1e-9


# ============================================================
# HELPERS
# ============================================================

def msg(text=""):
    print(text, flush=True)


def safe_float(value):
    if value is None:
        return 0.0
    try:
        if pd.isna(value):
            return 0.0
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return 0.0


def sanitize_name(name):
    name = re.sub(r"[^A-Za-z0-9_]", "_", name)
    if re.match(r"^[0-9]", name):
        name = "x_" + name
    return name


def get_k_suffix(layer_name):
    """
    For selected_dissolved_n2 -> n2
    For selected_dissolved_n3 -> n3
    For selected_dissolved_n4 -> n4
    """
    clean = layer_name.split(".")[-1]
    m = re.search(r"n(\d+)$", clean, re.IGNORECASE)
    if m:
        return f"n{m.group(1)}"
    return "n_unknown"


def list_layer_names(dataset_path):
    layers = pyogrio.list_layers(str(dataset_path))
    if hasattr(layers, "shape"):
        return [str(row[0]) for row in layers]
    return [str(row[0]) if isinstance(row, (list, tuple)) else str(row) for row in layers]


def normalize_layer_name(layer_name):
    """
    ArcGIS may display GeoPackage layers as main.layer_name.
    GDAL/pyogrio usually uses layer_name.
    This strips a leading main. if present.
    """
    layer_name = str(layer_name)
    if layer_name.lower().startswith("main."):
        return layer_name.split(".", 1)[1]
    return layer_name


def actual_layer_name(dataset_path, requested_layer_name):
    """
    Finds a layer case-insensitively and tolerates either:
        selected_dissolved_n2
        main.selected_dissolved_n2
    """
    requested_clean = normalize_layer_name(requested_layer_name).lower()

    for lyr in list_layer_names(dataset_path):
        lyr_clean = normalize_layer_name(lyr).lower()
        if lyr_clean == requested_clean:
            return lyr

    available = list_layer_names(dataset_path)
    raise RuntimeError(
        "Layer not found.\n"
        f"Requested: {requested_layer_name}\n"
        f"Dataset:   {dataset_path}\n"
        f"Available layers:\n  " + "\n  ".join(available)
    )


def layer_exists(dataset_path, layer_name):
    try:
        actual_layer_name(dataset_path, layer_name)
        return True
    except Exception:
        return False


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]
    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def read_vector(path, layer, columns=None, bbox=None, read_geometry=True):
    kwargs = {
        "layer": layer,
        "columns": columns,
        "bbox": bbox,
        "read_geometry": read_geometry,
    }
    kwargs = {k: v for k, v in kwargs.items() if v is not None}

    try:
        return pyogrio.read_dataframe(str(path), fid_as_index=True, **kwargs)
    except TypeError:
        return pyogrio.read_dataframe(str(path), **kwargs)


def add_stable_id_from_index(gdf, id_field):
    out = gdf.copy()
    out[id_field] = out.index.to_series(index=out.index).astype(str).values
    out = out.reset_index(drop=True)
    return out


def geometry_union(gdf):
    try:
        return gdf.geometry.union_all()
    except Exception:
        return gdf.geometry.unary_union


def make_valid_if_requested(gdf, label):
    if not fix_invalid_geometries:
        return gdf
    msg(f"    Repairing invalid geometries: {label}")
    out = gdf.copy()
    out["geometry"] = out.geometry.make_valid()
    return out


def is_projected_crs(gdf):
    try:
        return bool(gdf.crs and gdf.crs.is_projected)
    except Exception:
        return False


def ensure_same_crs(gdf, target_crs, label):
    if gdf.crs is None:
        raise RuntimeError(f"{label} has unknown CRS.")
    if target_crs is None:
        raise RuntimeError("Target layer has unknown CRS.")
    if gdf.crs != target_crs:
        msg(f"    Reprojecting {label} to match block layer CRS.")
        return gdf.to_crs(target_crs)
    return gdf


def write_gpkg_layer(gdf, gpkg_path, layer_name):
    gdf.to_file(
        str(gpkg_path),
        layer=layer_name,
        driver="GPKG",
        engine="pyogrio",
    )


def iter_selected_block_folders(base_folder):
    wanted = set(block_filter)
    for item in sorted(base_folder.iterdir()):
        if item.is_dir() and item.name in wanted:
            yield item


def get_beam_layers(input_gpkg):
    """
    Return actual layer names from the input GeoPackage for whichever
    requested selected_dissolved_n* layers exist.

    Missing requested layers are skipped instead of causing the whole
    block folder to fail.
    """
    actual_layers = []

    msg("Available layers in BEAM GeoPackage:")
    for lyr in list_layer_names(input_gpkg):
        msg(f"  {lyr}")

    for requested in beam_layer_names:
        try:
            actual = actual_layer_name(input_gpkg, requested)
            actual_layers.append(actual)
        except RuntimeError:
            msg(f"  WARNING: Requested BEAM layer not found; skipping: {requested}")

    if not actual_layers:
        raise RuntimeError(
            "None of the requested BEAM dissolved layers were found.\n"
            f"Requested layers:\n  " + "\n  ".join(beam_layer_names) + "\n"
            f"Dataset:\n  {input_gpkg}"
        )

    return actual_layers


def read_block_layers_for_folder(input_gpkg, layer_names):
    block_layers = {}

    for layer_name in layer_names:
        gdf = read_vector(input_gpkg, layer=layer_name)

        if gdf.empty:
            msg(f"  Warning: {layer_name} is empty.")

        gdf = gdf.reset_index(drop=True).copy()
        gdf["_block_fid"] = range(1, len(gdf) + 1)

        block_layers[layer_name] = gdf

    return block_layers


def combined_bounds(block_layers):
    bounds = []
    for gdf in block_layers.values():
        if not gdf.empty:
            bounds.append(gdf.total_bounds)

    if not bounds:
        return None

    b = pd.DataFrame(bounds, columns=["minx", "miny", "maxx", "maxy"])
    return (
        float(b["minx"].min()),
        float(b["miny"].min()),
        float(b["maxx"].max()),
        float(b["maxy"].max()),
    )


def read_population_candidates(folder_block_layers):
    first_gdf = next(iter(folder_block_layers.values()))
    target_crs = first_gdf.crs

    if target_crs is None:
        raise RuntimeError("Input BEAM block layer has unknown CRS.")

    bbox = combined_bounds(folder_block_layers)
    if bbox is None:
        raise RuntimeError("No non-empty BEAM block layers found for folder.")

    msg("  Reading candidate pop_grid cells from FileGDB...")
    msg(f"    bbox: {bbox}")

    pop_layer_actual = actual_layer_name(population_gdb, pop_grid_layer)

    pop_candidates = read_vector(
        population_gdb,
        layer=pop_layer_actual,
        columns=[pop_field],
        bbox=bbox,
    )

    if pop_candidates.empty:
        return pop_candidates, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    pop_candidates = ensure_same_crs(pop_candidates, target_crs, "pop_grid")
    require_columns(pop_candidates, [pop_field], "pop_grid")

    pop_candidates = add_stable_id_from_index(pop_candidates, "_grid_fid")

    all_blocks = pd.concat(
        [gdf[["geometry"]] for gdf in folder_block_layers.values() if not gdf.empty],
        ignore_index=True,
    )
    all_blocks = gpd.GeoDataFrame(all_blocks, geometry="geometry", crs=target_crs)
    all_blocks_union = geometry_union(all_blocks)

    pop_selected = pop_candidates[
        pop_candidates.geometry.intersects(all_blocks_union)
    ].copy()

    msg(f"  Candidate pop_grid cells read: {len(pop_candidates):,}")
    msg(f"  Selected pop_grid cells:       {len(pop_selected):,}")

    if pop_selected.empty:
        return pop_selected, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    grid_bbox = tuple(float(v) for v in pop_selected.total_bounds)

    msg("  Reading candidate buildings_inside features from FileGDB...")
    msg(f"    selected grid bbox: {grid_bbox}")

    bldg_layer_actual = actual_layer_name(population_gdb, buildings_layer)

    try:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            columns=[],
            bbox=grid_bbox,
        )
    except Exception:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            bbox=grid_bbox,
        )

    buildings = ensure_same_crs(buildings, target_crs, "buildings_inside")

    grid_union = geometry_union(pop_selected)
    buildings = buildings[
        buildings.geometry.intersects(grid_union)
    ].copy()

    buildings = buildings[["geometry"]].copy()

    msg(f"  Buildings selected:            {len(buildings):,}")

    return pop_selected, buildings


def write_assigned_population_csv(csv_path, block_fid_field, block_ids, block_assigned_population):
    if csv_path.exists():
        msg("Deleting existing CSV:")
        msg(f"  {csv_path}")
        csv_path.unlink()

    msg("Writing final block population CSV:")
    msg(f"  {csv_path}")

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([block_fid_field, "assigned_population"])

        for block_id in sorted(block_ids):
            assigned_pop = safe_float(block_assigned_population.get(int(block_id), 0.0))
            writer.writerow([block_id, assigned_pop])


def write_grid_detail_csv(csv_path, block_fid_field, grid_detail_rows):
    if csv_path.exists():
        msg("Deleting existing CSV:")
        msg(f"  {csv_path}")
        csv_path.unlink()

    msg("Writing grid apportionment detail CSV:")
    msg(f"  {csv_path}")

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "FID_pop_grid_selection",
            "grid_code",
            block_fid_field,
            "group_built_area_m2",
            "total_built_area_m2_in_grid",
            "built_area_share",
            "apportioned_population",
            "note",
        ])

        for r in grid_detail_rows:
            writer.writerow([
                r["FID_pop_grid_selection"],
                r["grid_code"],
                r["block_fid"],
                r["group_built_area_m2"],
                r["total_built_area_m2_in_grid"],
                r["built_area_share"],
                r["apportioned_population"],
                r["note"],
            ])


# ============================================================
# CORE POPULATION ASSIGNMENT
# ============================================================

def calculate_population_for_block_layer(blocks_gdf, block_layer_name, pop_selected, buildings, block_folder):
    safe_block_name = sanitize_name(normalize_layer_name(block_layer_name))
    k_suffix = get_k_suffix(block_layer_name)

    msg("")
    msg("------------------------------------------------------------")
    msg(f"Processing BEAM layer: {block_layer_name}")
    msg(f"  n suffix: {k_suffix}")
    msg("------------------------------------------------------------")

    assigned_pop_csv = block_folder / f"{safe_block_name}_assigned_population.csv"
    grid_detail_csv = block_folder / f"{safe_block_name}_grid_apportionment_detail.csv"

    blocks = blocks_gdf.copy()

    if blocks.empty:
        blocks[population_field] = []
        write_assigned_population_csv(assigned_pop_csv, "block_fid", [], {})
        write_grid_detail_csv(grid_detail_csv, "block_fid", [])
        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": 0,
            "identity_features": None,
            "building_intersections": 0,
            "block_features_updated": 0,
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": "empty block layer",
        }

    if not is_projected_crs(blocks):
        msg("  WARNING: block layer CRS does not appear to be projected.")
        msg("  Area calculations may not be in square meters.")

    blocks = make_valid_if_requested(blocks, "blocks")
    pop_selected = make_valid_if_requested(pop_selected, "pop_grid")
    buildings = make_valid_if_requested(buildings, "buildings")

    block_union = geometry_union(blocks[["geometry"]])

    pop_for_layer = pop_selected[
        pop_selected.geometry.intersects(block_union)
    ].copy()

    selected_count = len(pop_for_layer)

    msg(f"Selected pop_grid cells for this layer: {selected_count:,}")

    def finish_zero(status):
        blocks[population_field] = 0.0
        write_assigned_population_csv(
            assigned_pop_csv,
            block_fid_field="block_fid",
            block_ids=list(blocks["_block_fid"]),
            block_assigned_population={},
        )
        write_grid_detail_csv(
            grid_detail_csv,
            block_fid_field="block_fid",
            grid_detail_rows=[],
        )
        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": selected_count,
            "identity_features": None,
            "building_intersections": 0,
            "block_features_updated": len(blocks),
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": status,
        }

    if selected_count == 0:
        msg("WARNING: No pop_grid cells selected. Population will be set to 0.")
        return finish_zero("no pop_grid cells selected")

    layer_grid_union = geometry_union(pop_for_layer)
    buildings_for_layer = buildings[
        buildings.geometry.intersects(layer_grid_union)
    ].copy()

    if buildings_for_layer.empty:
        msg("WARNING: No buildings intersect selected pop_grid cells. Population will be set to 0.")
        return finish_zero("no buildings in selected pop_grid cells")

    msg("Running overlay: selected pop_grid cells ∩ buildings...")

    grid_bldg = gpd.overlay(
        pop_for_layer[["_grid_fid", pop_field, "geometry"]],
        buildings_for_layer[["geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if grid_bldg.empty:
        msg("WARNING: Grid-building overlay produced no features. Population will be set to 0.")
        return finish_zero("no grid-building intersections")

    grid_bldg[area_field] = grid_bldg.geometry.area
    grid_bldg = grid_bldg[grid_bldg[area_field] > area_tolerance].copy()

    building_intersections_count = len(grid_bldg)

    msg(f"Grid-building intersection features: {building_intersections_count:,}")

    if grid_bldg.empty:
        return finish_zero("all grid-building intersections had zero area")

    grid_total_built_area = (
        grid_bldg.groupby("_grid_fid", dropna=False)[area_field]
        .sum()
        .to_dict()
    )

    grid_population = (
        grid_bldg.groupby("_grid_fid", dropna=False)[pop_field]
        .first()
        .apply(safe_float)
        .to_dict()
    )

    msg("Running overlay: grid-building pieces ∩ block polygons...")

    block_bldg = gpd.overlay(
        grid_bldg[["_grid_fid", "geometry"]],
        blocks[["_block_fid", "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if block_bldg.empty:
        block_bldg[area_field] = []
    else:
        block_bldg[area_field] = block_bldg.geometry.area
        block_bldg = block_bldg[block_bldg[area_field] > area_tolerance].copy()

    msg(f"Grid-building-block intersection features: {len(block_bldg):,}")

    if block_bldg.empty:
        grid_block_built_area = pd.DataFrame(columns=["_grid_fid", "_block_fid", area_field])
    else:
        grid_block_built_area = (
            block_bldg.groupby(["_grid_fid", "_block_fid"], dropna=False)[area_field]
            .sum()
            .reset_index()
        )

    msg("Apportioning grid-cell population to block features...")

    block_assigned_population = defaultdict(float)
    grid_detail_rows = []

    inside_area_by_grid = defaultdict(dict)
    inside_total_by_grid = defaultdict(float)

    for _, r in grid_block_built_area.iterrows():
        grid_id = r["_grid_fid"]
        block_fid = int(r["_block_fid"])
        group_area = safe_float(r[area_field])
        inside_area_by_grid[grid_id][block_fid] = group_area
        inside_total_by_grid[grid_id] += group_area

    zero_area_grid_count = 0

    for grid_id in sorted(grid_total_built_area.keys(), key=lambda x: str(x)):
        total_area = safe_float(grid_total_built_area[grid_id])
        grid_pop = safe_float(grid_population.get(grid_id, 0.0))

        if total_area <= 0:
            zero_area_grid_count += 1
            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": None,
                "group_built_area_m2": 0.0,
                "total_built_area_m2_in_grid": 0.0,
                "built_area_share": 0.0,
                "apportioned_population": 0.0,
                "note": "zero total built area in grid",
            })
            continue

        for block_fid, group_area in sorted(inside_area_by_grid.get(grid_id, {}).items()):
            built_area_share = group_area / total_area
            apportioned_pop = grid_pop * built_area_share

            block_assigned_population[block_fid] += apportioned_pop

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": block_fid,
                "group_built_area_m2": group_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": built_area_share,
                "apportioned_population": apportioned_pop,
                "note": "inside block",
            })

        outside_area = total_area - safe_float(inside_total_by_grid.get(grid_id, 0.0))

        if outside_area > area_tolerance:
            outside_share = outside_area / total_area
            outside_pop = grid_pop * outside_share

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": -1,
                "group_built_area_m2": outside_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": outside_share,
                "apportioned_population": outside_pop,
                "note": "outside blocks",
            })

    msg(f"Grid cells with zero total built area: {zero_area_grid_count:,}")
    msg(f"Block features receiving population: {len(block_assigned_population):,}")

    write_assigned_population_csv(
        assigned_pop_csv,
        block_fid_field="block_fid",
        block_ids=list(blocks["_block_fid"]),
        block_assigned_population=block_assigned_population,
    )

    write_grid_detail_csv(
        grid_detail_csv,
        block_fid_field="block_fid",
        grid_detail_rows=grid_detail_rows,
    )

    blocks[population_field] = blocks["_block_fid"].map(
        lambda x: float(block_assigned_population.get(int(x), 0.0))
    )

    updated_count = len(blocks)
    zero_count = int((blocks[population_field] == 0).sum())

    total_grid_population_seen = sum(safe_float(v) for v in grid_population.values())
    total_population_assigned_to_blocks = sum(safe_float(v) for v in block_assigned_population.values())

    msg("Layer done.")
    msg("Summary:")
    msg(f"  Block layer:                         {block_layer_name}")
    msg(f"  Selected grid cells:                 {selected_count:,}")
    msg(f"  Identity features:                   not created in GeoPandas version")
    msg(f"  Grid-building intersections:         {building_intersections_count:,}")
    msg(f"  Grid-building-block intersections:   {len(block_bldg):,}")
    msg(f"  Block features updated:              {updated_count:,}")
    msg(f"  Block features assigned zero pop:    {zero_count:,}")
    msg(f"  Total grid population represented:   {total_grid_population_seen}")
    msg(f"  Total population assigned to blocks: {total_population_assigned_to_blocks}")

    return blocks, {
        "block_layer": normalize_layer_name(block_layer_name),
        "k": k_suffix,
        "selected_grid_cells": selected_count,
        "identity_features": None,
        "building_intersections": building_intersections_count,
        "block_features_updated": updated_count,
        "total_grid_population_seen": total_grid_population_seen,
        "total_population_assigned_to_blocks": total_population_assigned_to_blocks,
        "status": "success",
    }


# ============================================================
# MAIN
# ============================================================

def main():
    msg("Starting BEAM no-ArcPy population assignment")
    msg(f"Block folder root: {large_pop_blocks_folder}")
    msg(f"Population GDB:    {population_gdb}")
    msg(f"Input GPKG name:   {input_blocks_gpkg_name}")
    msg(f"Output GPKG name:  {populated_blocks_gpkg_name}")
    msg("")

    if not large_pop_blocks_folder.is_dir():
        raise FileNotFoundError(
            "large_pop_blocks_folder does not exist:\n"
            f"{large_pop_blocks_folder}"
        )

    if not population_gdb.exists():
        raise FileNotFoundError(
            "Population geodatabase does not exist:\n"
            f"{population_gdb}"
        )

    if not layer_exists(population_gdb, pop_grid_layer):
        raise FileNotFoundError(
            f"Population grid layer '{pop_grid_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    if not layer_exists(population_gdb, buildings_layer):
        raise FileNotFoundError(
            f"Buildings layer '{buildings_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    block_folders = list(iter_selected_block_folders(large_pop_blocks_folder))

    found_names = {p.name for p in block_folders}
    missing = [b for b in block_filter if b not in found_names]

    msg("Selected block folders found:")
    msg(f"  {len(block_folders)} of {len(block_filter)}")

    if missing:
        msg("WARNING: These requested folders were not found:")
        for b in missing:
            msg(f"  {b}")

    overall_summary = []

    for block_folder in block_folders:
        block_folder_name = block_folder.name

        msg("")
        msg("============================================================")
        msg(f"Block folder: {block_folder_name}")
        msg("============================================================")

        input_blocks_gpkg = block_folder / input_blocks_gpkg_name
        populated_blocks_gpkg = block_folder / populated_blocks_gpkg_name

        if not input_blocks_gpkg.exists():
            msg("Skipping folder because BEAM input GeoPackage does not exist:")
            msg(f"  {input_blocks_gpkg}")
            overall_summary.append({
                "block_folder": block_folder_name,
                "block_layer": None,
                "k": None,
                "selected_grid_cells": None,
                "identity_features": None,
                "building_intersections": None,
                "block_features_updated": None,
                "total_grid_population_seen": None,
                "total_population_assigned_to_blocks": None,
                "status": "ERROR: input graph_boundary_penalty_partitions_beam.gpkg not found",
            })
            continue

        try:
            block_layer_names = get_beam_layers(input_blocks_gpkg)

            msg("BEAM block layers found:")
            for layer_name in block_layer_names:
                msg(f"  {layer_name}")

            if overwrite_outputs and populated_blocks_gpkg.exists():
                msg("Deleting existing populated BEAM output GeoPackage:")
                msg(f"  {populated_blocks_gpkg}")
                populated_blocks_gpkg.unlink()

            block_layers = read_block_layers_for_folder(
                input_blocks_gpkg,
                block_layer_names,
            )

            pop_selected, buildings = read_population_candidates(block_layers)

            for layer_name in block_layer_names:
                try:
                    updated_gdf, result = calculate_population_for_block_layer(
                        blocks_gdf=block_layers[layer_name],
                        block_layer_name=layer_name,
                        pop_selected=pop_selected,
                        buildings=buildings,
                        block_folder=block_folder,
                    )

                    if "_block_fid" in updated_gdf.columns:
                        updated_gdf_to_write = updated_gdf.drop(columns=["_block_fid"]).copy()
                    else:
                        updated_gdf_to_write = updated_gdf.copy()

                    # All layers produced by this notebook are BEAM-derived candidates.

                    # Write output using clean layer names: selected_dissolved_n2, etc.
                    output_layer_name = normalize_layer_name(layer_name)

                    write_gpkg_layer(
                        updated_gdf_to_write,
                        populated_blocks_gpkg,
                        output_layer_name,
                    )

                    msg("Populated BEAM block layer written to:")
                    msg(f"  {populated_blocks_gpkg} | {output_layer_name}")

                    result["block_folder"] = block_folder_name
                    result["block_layer"] = output_layer_name
                    overall_summary.append(result)

                except Exception as e:
                    msg("")
                    msg("ERROR while processing:")
                    msg(f"  Folder: {block_folder_name}")
                    msg(f"  Layer:  {layer_name}")
                    msg(str(e))

                    overall_summary.append({
                        "block_folder": block_folder_name,
                        "block_layer": normalize_layer_name(layer_name),
                        "k": get_k_suffix(layer_name),
                        "selected_grid_cells": None,
                        "identity_features": None,
                        "building_intersections": None,
                        "block_features_updated": None,
                        "total_grid_population_seen": None,
                        "total_population_assigned_to_blocks": None,
                        "status": f"ERROR: {str(e)}",
                    })

        except Exception:
            msg("")
            msg("FAILED on this block folder:")
            msg(traceback.format_exc())

            overall_summary.append({
                "block_folder": block_folder_name,
                "block_layer": None,
                "k": None,
                "selected_grid_cells": None,
                "identity_features": None,
                "building_intersections": None,
                "block_features_updated": None,
                "total_grid_population_seen": None,
                "total_population_assigned_to_blocks": None,
                "status": "ERROR at folder level",
            })

    summary_csv = large_pop_blocks_folder / "new_blocks_population_assignment_summary_beam_no_arcpy.csv"

    if summary_csv.exists():
        msg("Deleting existing BEAM overall summary CSV:")
        msg(f"  {summary_csv}")
        summary_csv.unlink()

    msg("")
    msg("Writing BEAM overall summary CSV:")
    msg(f"  {summary_csv}")

    with open(summary_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "block_folder",
                "block_layer",
                "k",
                "selected_grid_cells",
                "identity_features",
                "building_intersections",
                "block_features_updated",
                "total_grid_population_seen",
                "total_population_assigned_to_blocks",
                "status",
            ],
        )

        writer.writeheader()

        for row in overall_summary:
            writer.writerow(row)

    msg("")
    msg("All done.")
    msg(f"Layers processed: {len(overall_summary)}")
    msg("Summary written to:")
    msg(f"  {summary_csv}")


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
r"""
Select one populated BEAM new-block layer per source block, without ArcPy.

This version processes the manually selected BEAM block folders defined in BLOCK_FOLDERS_TO_PROCESS.

Input per block folder:
    E:\_johannesburg\_analysis\heterogeneous_largePop_blocks\_<id>\new_blocks_populated_beam.gpkg

Expected layers:
    selected_dissolved_n2
    selected_dissolved_n3
    selected_dissolved_n4

Output per block folder:
    E:\_johannesburg\_analysis\heterogeneous_largePop_selection\_<id>\new_blocks_populated.gpkg

If a Step 9 canonical selection already exists, it is preserved once as:
    new_blocks_populated_pre_beam.gpkg

The canonical output GeoPackage contains one selected BEAM layer for that block.

Selection rules
---------------
For LargePop == 1:
    - Select the first layer, checking n=2, then n=3, then n=4,
      where all feature populations are below 1000.
    - If none pass the population rule, select n=4 if all n=4 feature areas
      are below 100,000 m2.
    - If n=4 also fails the area rule, still select n=4 and log a warning.

For non-LargePop heterogeneous blocks:
    - Select the first layer, checking n=2, then n=3, then n=4,
      where all feature areas are below 100,000 m2.
    - If none pass the area rule, still select n=4 and log a warning.
"""

import traceback
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd
import geopandas as gpd
import pyogrio


# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------

INPUT_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"
)

OUTPUT_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection"
)

SOURCE_BLOCKS_GDB = Path(
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
)

SOURCE_BLOCKS_LAYER = "heterogeneous_largePop_blocks"

INPUT_GPKG_NAME = "new_blocks_populated_beam.gpkg"
CANONICAL_GPKG_NAME = "new_blocks_populated.gpkg"
PRE_BEAM_GPKG_NAME = "new_blocks_populated_pre_beam.gpkg"

BLOCK_FOLDERS_TO_PROCESS = [
    "_52730",
    "_56219",
    "_62733",
    "_71608",
    "_72054",
    "_76290",
    "_83562",
    "_84830",
    "_89704"
]

N_VALUES = [2, 3, 4]

BLOCK_ID_FIELD = "block_id"
HETERO_FIELDS = ["HH_CC", "HH_Grtr10ha", "CC_Grtr10ha"]
LARGEPOP_FIELD = "LargePop"

POPULATION_FIELD = "population"

# Candidate-block area used for the 100,000 m2 threshold.
# Step 11 explicitly sums tessellation cell_area_m2 into each dissolved BEAM part.
AREA_FIELD_CANDIDATES = [
    "cell_area_m2",
]

POPULATION_THRESHOLD = 1000.0
AREA_THRESHOLD_M2 = 100000.0

OVERWRITE_OUTPUTS = True

LOG_PATH = OUTPUT_ROOT / "beam_selection_log.txt"
SUMMARY_CSV = OUTPUT_ROOT / "beam_selection_summary.csv"


# ------------------------------------------------------------
# Logging and helpers
# ------------------------------------------------------------

def log(message="", also_print=True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    if also_print:
        print(message, flush=True)


def list_layer_names(dataset_path):
    layers = pyogrio.list_layers(str(dataset_path))
    if hasattr(layers, "shape"):
        return [str(row[0]) for row in layers]
    return [str(row[0]) if isinstance(row, (list, tuple)) else str(row) for row in layers]


def layer_base_name(layer_name):
    # Handles names displayed as main.selected_dissolved_n2.
    return str(layer_name).split(".")[-1]


def normalize_layer_name(layer_name):
    return layer_base_name(layer_name)


def layer_exists(dataset_path, layer_name):
    target = normalize_layer_name(layer_name).lower()
    return target in {normalize_layer_name(layer).lower() for layer in list_layer_names(dataset_path)}


def actual_layer_name(dataset_path, layer_name):
    target = normalize_layer_name(layer_name).lower()
    for layer in list_layer_names(dataset_path):
        if normalize_layer_name(layer).lower() == target:
            return layer
    raise RuntimeError(
        f"Layer not found: {layer_name} in {dataset_path}\n"
        f"Available layers: {list_layer_names(dataset_path)}"
    )


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]
    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def actual_column_name(df, requested_name):
    requested_lower = requested_name.lower()
    for col in df.columns:
        if col.lower() == requested_lower:
            return col
    raise RuntimeError(f"Column not found: {requested_name}")


def first_existing_column(df, candidates, label):
    lower_lookup = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_lookup:
            return lower_lookup[cand.lower()]
    raise RuntimeError(
        f"{label} is missing an area column. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def flag_is_one(value):
    try:
        return int(float(value)) == 1
    except Exception:
        return False


def safe_float(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return None


def block_folder_to_lookup_block_id(block_folder_name):
    return "blk_" + block_folder_name.lstrip("_")


def parse_n_from_layer(layer_name):
    base = layer_base_name(layer_name)
    last = base.split("_")[-1]
    if last.lower().startswith("n") and last[1:].isdigit():
        return int(last[1:])
    return None


def find_beam_layers_by_n(gpkg_path):
    out = {}

    for layer in list_layer_names(gpkg_path):
        base = layer_base_name(layer).lower()

        if not base.startswith("selected_dissolved_n"):
            continue

        n = parse_n_from_layer(base)

        if n in N_VALUES:
            out[n] = layer

    return out


# ------------------------------------------------------------
# Source membership lookup
# ------------------------------------------------------------

def build_hetero_orig(row):
    active = []
    for field in HETERO_FIELDS:
        if field in row.index and flag_is_one(row[field]):
            active.append(field)
    return "none" if not active else "|".join(active)


def read_source_membership_lookup():
    if not SOURCE_BLOCKS_GDB.exists():
        raise FileNotFoundError(f"Missing source GDB: {SOURCE_BLOCKS_GDB}")

    if not layer_exists(SOURCE_BLOCKS_GDB, SOURCE_BLOCKS_LAYER):
        raise FileNotFoundError(
            f"Missing source layer {SOURCE_BLOCKS_LAYER} in {SOURCE_BLOCKS_GDB}"
        )

    actual_layer = actual_layer_name(SOURCE_BLOCKS_GDB, SOURCE_BLOCKS_LAYER)
    columns = [BLOCK_ID_FIELD] + HETERO_FIELDS + [LARGEPOP_FIELD]

    log("Reading source block membership lookup...")
    log(f"  {SOURCE_BLOCKS_GDB} | {actual_layer}")

    source_df = pyogrio.read_dataframe(
        str(SOURCE_BLOCKS_GDB),
        layer=actual_layer,
        columns=columns,
        read_geometry=False,
    )

    require_columns(source_df, columns, SOURCE_BLOCKS_LAYER)

    lookup = {}
    duplicate_count = 0

    for _, row in source_df.iterrows():
        block_id = str(row[BLOCK_ID_FIELD]).strip()

        if not block_id:
            continue

        if block_id in lookup:
            duplicate_count += 1

        record = {field: 1 if flag_is_one(row[field]) else 0 for field in HETERO_FIELDS}
        record[LARGEPOP_FIELD] = 1 if flag_is_one(row[LARGEPOP_FIELD]) else 0
        record["hetero_orig"] = build_hetero_orig(row)
        record["large_pop_orig"] = record[LARGEPOP_FIELD]

        lookup[block_id] = record

    log(f"  Source records read: {len(source_df):,}")
    log(f"  Lookup records:      {len(lookup):,}")

    if duplicate_count:
        log(f"  WARNING: duplicate block_id values found: {duplicate_count:,}")

    return lookup


# ------------------------------------------------------------
# Layer evaluation and selection
# ------------------------------------------------------------

def evaluate_layer(gpkg_path, layer_name):
    gdf = gpd.read_file(str(gpkg_path), layer=layer_name)

    require_columns(gdf, [POPULATION_FIELD], layer_name)

    pop_col = actual_column_name(gdf, POPULATION_FIELD)

    area_col = first_existing_column(
        gdf,
        AREA_FIELD_CANDIDATES,
        layer_name,
    )
    areas = gdf[area_col].map(safe_float)

    populations = gdf[pop_col].map(safe_float)

    nonnull_populations = populations.dropna()
    nonnull_areas = areas.dropna()
    n_features = len(gdf)

    if n_features == 0:
        return {
            "layer_name": layer_name,
            "area_field_used": area_col,
            "n_features": 0,
            "max_population": None,
            "max_area_m2": None,
            "null_population_count": 0,
            "null_area_count": 0,
            "all_population_below_1000": False,
            "all_area_below_100000": False,
        }

    null_population_count = int(populations.isna().sum())
    null_area_count = int(areas.isna().sum())

    all_pop_below = (
        null_population_count == 0
        and bool((nonnull_populations < POPULATION_THRESHOLD).all())
    )

    all_area_below = (
        null_area_count == 0
        and bool((nonnull_areas < AREA_THRESHOLD_M2).all())
    )

    return {
        "layer_name": layer_name,
        "area_field_used": area_col,
        "n_features": n_features,
        "max_population": float(nonnull_populations.max()) if len(nonnull_populations) else None,
        "max_area_m2": float(nonnull_areas.max()) if len(nonnull_areas) else None,
        "null_population_count": null_population_count,
        "null_area_count": null_area_count,
        "all_population_below_1000": all_pop_below,
        "all_area_below_100000": all_area_below,
    }


def choose_layer_for_block(gpkg_path, layer_by_n, membership):
    evaluations = {}

    for n in N_VALUES:
        if n not in layer_by_n:
            evaluations[n] = {"layer_name": None, "missing": True}
            continue

        stats = evaluate_layer(gpkg_path, layer_by_n[n])
        stats["missing"] = False
        evaluations[n] = stats

    large_pop = membership[LARGEPOP_FIELD] == 1
    hetero_any = any(membership[f] == 1 for f in HETERO_FIELDS)

    if large_pop:
        for n in N_VALUES:
            stats = evaluations.get(n, {})
            if not stats.get("missing") and stats["all_population_below_1000"]:
                return (
                    n,
                    stats["layer_name"],
                    f"LargePop=1; selected first n where all feature populations are < {POPULATION_THRESHOLD:g}.",
                    "",
                    evaluations,
                )

        stats4 = evaluations.get(4, {})

        if not stats4.get("missing") and stats4.get("all_area_below_100000"):
            return (
                4,
                stats4["layer_name"],
                f"LargePop=1; no n=2..4 layer had all populations < {POPULATION_THRESHOLD:g}; selected n=4 because all feature areas are < {AREA_THRESHOLD_M2:g} m2.",
                "",
                evaluations,
            )

        if not stats4.get("missing"):
            return (
                4,
                stats4["layer_name"],
                f"LargePop=1; no n=2..4 layer had all populations < {POPULATION_THRESHOLD:g}, and n=4 did not have all areas < {AREA_THRESHOLD_M2:g} m2; selected n=4 anyway.",
                "LargePop fallback warning: n=4 violates both preferred population and fallback area criteria.",
                evaluations,
            )

        available = [n for n in N_VALUES if not evaluations.get(n, {}).get("missing")]

        if available:
            n = max(available)
            stats = evaluations[n]
            return (
                n,
                stats["layer_name"],
                f"LargePop=1; n=4 layer missing, so selected highest available n={n} as emergency fallback.",
                "Missing n=4 fallback warning.",
                evaluations,
            )

        return (
            None,
            None,
            "LargePop=1 but no candidate selected_dissolved_n* layers were available.",
            "No layers available.",
            evaluations,
        )

    if hetero_any:
        for n in N_VALUES:
            stats = evaluations.get(n, {})
            if not stats.get("missing") and stats["all_area_below_100000"]:
                return (
                    n,
                    stats["layer_name"],
                    f"LargePop!=1 and heterogeneous; selected first n where all feature areas are < {AREA_THRESHOLD_M2:g} m2.",
                    "",
                    evaluations,
                )

        stats4 = evaluations.get(4, {})

        if not stats4.get("missing"):
            return (
                4,
                stats4["layer_name"],
                f"LargePop!=1 and heterogeneous; no n=2..4 layer had all feature areas < {AREA_THRESHOLD_M2:g} m2; selected n=4 anyway.",
                "Heterogeneous fallback warning: n=4 violates area criterion.",
                evaluations,
            )

        available = [n for n in N_VALUES if not evaluations.get(n, {}).get("missing")]

        if available:
            n = max(available)
            stats = evaluations[n]
            return (
                n,
                stats["layer_name"],
                f"LargePop!=1 and heterogeneous; n=4 layer missing, so selected highest available n={n} as emergency fallback.",
                "Missing n=4 fallback warning.",
                evaluations,
            )

        return (
            None,
            None,
            "Heterogeneous non-LargePop block but no candidate selected_dissolved_n* layers were available.",
            "No layers available.",
            evaluations,
        )

    return (
        None,
        None,
        "Block is neither LargePop nor heterogeneous according to lookup; skipped.",
        "Skipped: no relevant origin membership.",
        evaluations,
    )


# ------------------------------------------------------------
# Output helpers
# ------------------------------------------------------------

def write_selected_beam_to_canonical(
    input_gpkg,
    input_layer,
    output_block_folder,
    output_layer,
):
    """
    Preserve the Step 9 canonical selection once, then replace the canonical
    new_blocks_populated.gpkg with the selected BEAM layer.
    """
    output_block_folder.mkdir(parents=True, exist_ok=True)

    canonical_gpkg = output_block_folder / CANONICAL_GPKG_NAME
    pre_beam_gpkg = output_block_folder / PRE_BEAM_GPKG_NAME

    if canonical_gpkg.exists() and not pre_beam_gpkg.exists():
        shutil.copy2(canonical_gpkg, pre_beam_gpkg)
        log(f"  Preserved pre-BEAM selection: {pre_beam_gpkg}")

    gdf = gpd.read_file(str(input_gpkg), layer=input_layer)

    if canonical_gpkg.exists():
        if OVERWRITE_OUTPUTS:
            canonical_gpkg.unlink()
        else:
            raise FileExistsError(
                f"Canonical output already exists and OVERWRITE_OUTPUTS=False: {canonical_gpkg}"
            )

    gdf.to_file(
        str(canonical_gpkg),
        layer=output_layer,
        driver="GPKG",
        engine="pyogrio",
    )

    return canonical_gpkg


def evaluation_value(evaluations, n, key):
    stats = evaluations.get(n, {})
    if stats.get("missing"):
        return None
    return stats.get(key)


def add_evaluation_fields(row, evaluations):
    for n in N_VALUES:
        row[f"n{n}_n_features"] = evaluation_value(evaluations, n, "n_features")
        row[f"n{n}_area_field_used"] = evaluation_value(evaluations, n, "area_field_used")
        row[f"n{n}_max_population"] = evaluation_value(evaluations, n, "max_population")
        row[f"n{n}_max_area_m2"] = evaluation_value(evaluations, n, "max_area_m2")
        row[f"n{n}_all_population_below_1000"] = evaluation_value(evaluations, n, "all_population_below_1000")
        row[f"n{n}_all_area_below_100000"] = evaluation_value(evaluations, n, "all_area_below_100000")
    return row


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if LOG_PATH.exists():
        LOG_PATH.unlink()

    log("Starting BEAM new-block selection workflow")
    log(f"Input root:  {INPUT_ROOT}")
    log(f"Output root: {OUTPUT_ROOT}")
    log(f"Input GPKG:  {INPUT_GPKG_NAME}")
    log(f"Canonical selected GPKG: {CANONICAL_GPKG_NAME}")
    log("")

    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Input root does not exist: {INPUT_ROOT}")

    source_lookup = read_source_membership_lookup()

    summary_rows = []

    requested_folders = [INPUT_ROOT / name for name in BLOCK_FOLDERS_TO_PROCESS]

    for block_folder in requested_folders:
        block_folder_name = block_folder.name
        lookup_block_id = block_folder_to_lookup_block_id(block_folder_name)
        input_gpkg = block_folder / INPUT_GPKG_NAME
        output_block_folder = OUTPUT_ROOT / block_folder_name

        log("=" * 80)
        log(f"Block folder: {block_folder_name}")
        log(f"Lookup block_id: {lookup_block_id}")

        row_base = {
            "block_folder": block_folder_name,
            "lookup_block_id": lookup_block_id,
            "input_gpkg": str(input_gpkg),
            "output_folder": str(output_block_folder),
        }

        try:
            if not block_folder.exists():
                reason = f"Missing input block folder: {block_folder}"
                log(f"  SKIP: {reason}")
                summary_rows.append({
                    **row_base,
                    "status": "skipped",
                    "selected_n": None,
                    "selected_layer": None,
                    "selection_reason": reason,
                    "warning": "missing input block folder",
                })
                continue

            if not input_gpkg.exists():
                reason = f"Missing input GeoPackage: {input_gpkg}"
                log(f"  SKIP: {reason}")
                summary_rows.append({
                    **row_base,
                    "status": "skipped",
                    "selected_n": None,
                    "selected_layer": None,
                    "selection_reason": reason,
                    "warning": "missing input gpkg",
                })
                continue

            if lookup_block_id not in source_lookup:
                reason = f"block_id {lookup_block_id} not found in source lookup"
                log(f"  SKIP: {reason}")
                summary_rows.append({
                    **row_base,
                    "status": "skipped",
                    "selected_n": None,
                    "selected_layer": None,
                    "selection_reason": reason,
                    "warning": "missing source lookup record",
                })
                continue

            membership = source_lookup[lookup_block_id]

            log(
                "  Membership: "
                f"HH_CC={membership['HH_CC']}, "
                f"HH_Grtr10ha={membership['HH_Grtr10ha']}, "
                f"CC_Grtr10ha={membership['CC_Grtr10ha']}, "
                f"LargePop={membership['LargePop']}, "
                f"hetero_orig={membership['hetero_orig']}"
            )

            layer_by_n = find_beam_layers_by_n(input_gpkg)
            log(f"  Candidate BEAM layers found by n: {sorted(layer_by_n.keys())}")

            selected_n, selected_layer, reason, warning, evaluations = choose_layer_for_block(
                gpkg_path=input_gpkg,
                layer_by_n=layer_by_n,
                membership=membership,
            )

            if selected_layer is None:
                log(f"  SKIP: {reason}")
                if warning:
                    log(f"  WARNING: {warning}")

                row = {
                    **row_base,
                    "status": "skipped",
                    "selected_n": None,
                    "selected_layer": None,
                    "selection_reason": reason,
                    "warning": warning,
                    "HH_CC": membership["HH_CC"],
                    "HH_Grtr10ha": membership["HH_Grtr10ha"],
                    "CC_Grtr10ha": membership["CC_Grtr10ha"],
                    "LargePop": membership["LargePop"],
                    "hetero_orig": membership["hetero_orig"],
                    "large_pop_orig": membership["large_pop_orig"],
                }
                summary_rows.append(add_evaluation_fields(row, evaluations))
                continue

            log(f"  Selected n={selected_n}: {selected_layer}")
            log(f"  Reason: {reason}")

            if warning:
                log(f"  WARNING: {warning}")

            output_layer_name = normalize_layer_name(selected_layer)

            output_gpkg = write_selected_beam_to_canonical(
                input_gpkg=input_gpkg,
                input_layer=selected_layer,
                output_block_folder=output_block_folder,
                output_layer=output_layer_name,
            )

            log(f"  Wrote selected BEAM layer to canonical selection: {output_gpkg}")

            row = {
                **row_base,
                "status": "selected",
                "selected_n": selected_n,
                "selected_layer": output_layer_name,
                "output_gpkg": str(output_gpkg),
                "selection_reason": reason,
                "warning": warning,
                "HH_CC": membership["HH_CC"],
                "HH_Grtr10ha": membership["HH_Grtr10ha"],
                "CC_Grtr10ha": membership["CC_Grtr10ha"],
                "LargePop": membership["LargePop"],
                "hetero_orig": membership["hetero_orig"],
                "large_pop_orig": membership["large_pop_orig"],
            }

            summary_rows.append(add_evaluation_fields(row, evaluations))

        except Exception:
            error_text = traceback.format_exc()
            log("  ERROR:")
            log(error_text)

            summary_rows.append({
                **row_base,
                "status": "error",
                "selected_n": None,
                "selected_layer": None,
                "selection_reason": "error while processing block",
                "warning": error_text,
            })

    log("")
    log("Writing BEAM selection summary CSV...")
    log(f"  {SUMMARY_CSV}")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8")

    selected_count = int((summary_df["status"] == "selected").sum()) if "status" in summary_df.columns else 0
    skipped_count = int((summary_df["status"] == "skipped").sum()) if "status" in summary_df.columns else 0
    error_count = int((summary_df["status"] == "error").sum()) if "status" in summary_df.columns else 0

    log("")
    log("Finished.")
    log(f"Blocks processed: {len(summary_rows):,}")
    log(f"Selected blocks:  {selected_count:,}")
    log(f"Skipped blocks:   {skipped_count:,}")
    log(f"Errored blocks:   {error_count:,}")


if __name__ == "__main__":
    main()
